# Smart Parking Lot Optimization Using Linear Algebra

**Course:** UE25MA242A – Mathematical Foundation for AI & Data Science (MFAD 2026)

### Team members
- Arya B — PES1UG25AM472
- Aryan R — PES1UG25AM473
- Hisham — PES1UG25AM488
- Chirag P — PES1UG25AM481

## Purpose of this notebook
This notebook is the **live mathematical demonstration** for the project. It uses the same `parking_engine.py` used by the desktop application.

### Demo pipeline
**Parking data → matrix X → vehicle vector v → hard constraints → orthogonal projection → least-squares ranking → final parking assignment → visualization**

> Run the notebook from top to bottom. After each important output, explain what the output means rather than only showing the final answer.


## 1. Upload the final project ZIP

The notebook imports the same mathematical engine used by the project. Upload the ZIP folder that contains `src/parking_engine.py`.


In [ ]:
# 1. Automatic Local / Colab Setup
import os, sys

# Check if running locally or cloned repo with src/ available
local_engine = False
for path_candidate in ['src', '../src', 'MATH_SMART_PARKING_PROJECT/MFAD_Smart_Parking_SUBMISSION/src']:
    if os.path.exists(os.path.join(path_candidate, 'parking_engine.py')):
        sys.path.insert(0, os.path.abspath(path_candidate))
        print(f'Loaded parking_engine from: {path_candidate}')
        local_engine = True
        break

if not local_engine:
    try:
        from google.colab import files
        print('Colab detected. Please upload the project ZIP file:')
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if zip_names:
            import zipfile
            extract_root = '/content/MFAD_Smart_Parking_Project'
            os.makedirs(extract_root, exist_ok=True)
            with zipfile.ZipFile(zip_names[0], 'r') as z:
                z.extractall(extract_root)
            for base, dirs, files_in_dir in os.walk(extract_root):
                if 'parking_engine.py' in files_in_dir:
                    sys.path.insert(0, base)
                    print(f'Engine loaded from: {base}')
                    break
    except ImportError:
        print('Running outside Colab. Ensure src/parking_engine.py is accessible.')


In [ ]:
# 1b. Verify engine import
try:
    import parking_engine
    print('parking_engine imported successfully!')
except ImportError:
    print('Warning: Could not import parking_engine yet. Check src directory.')


## 2. Import the project engine

The next cell loads NumPy, pandas and Matplotlib for the demonstration and imports the **same `SmartParkingEngine` and `Vehicle` classes used by the desktop application**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from parking_engine import SmartParkingEngine, Vehicle

engine = SmartParkingEngine()
print("Smart Parking engine loaded successfully.")
print("Number of parking spaces:", len(engine.spots))
print("Entry gates:", list(engine.gates.keys()))


## 3. Stage 1 — Matrix representation

Each parking spot is represented by five normalized features:

`[dx, dy, EV, Accessible, SizeCapacity]`

**One row = one parking spot.**
The complete table is the parking feature matrix `X`.


In [ ]:
X_all = engine.build_feature_matrix("G2")
layout_df = pd.DataFrame(
    X_all,
    index=[s.spot_id for s in engine.spots],
    columns=["dx", "dy", "EV", "Accessible", "SizeCapacity"]
)

print("Matrix shape:", X_all.shape)
display(layout_df.round(3))


### What to say

> “The parking lot is converted into a matrix. Each row represents one parking space and each column represents a feature such as location, EV capability, accessibility and capacity.”


## 4. Stage 2 — Incoming vehicle vector

For the main demonstration, we use a vehicle with these requirements:

- Entry gate: **G2**
- EV charging: **Required**
- Accessible spot: **Required**
- Size: **Large**

The goal is to see how those real-world requirements become a numerical vector. This scenario is deliberately chosen so that **four** parking spaces satisfy all hard constraints, allowing us to visibly demonstrate the optimization/ranking step.


In [ ]:
vehicle = Vehicle(
    vehicle_id="DEMO-01",
    entry_gate="G2",
    ev_required=True,
    accessible_required=True,
    size="Large",
)

vehicle_vector = engine.build_vehicle_vector(vehicle)
print(vehicle)
print("\nVehicle demand vector v:")
print(np.round(vehicle_vector, 3))


### What to say

> “The arriving vehicle is represented in the same feature space. The vector therefore gives the mathematical description of what the vehicle needs.”


## 5. Stage 3 — Hard constraint filtering

Before optimization, the system removes invalid candidates:

- occupied spaces;
- non-EV spaces when EV is required;
- non-accessible spaces when accessibility is required;
- spaces too small for the vehicle.

This prevents the optimizer from returning a mathematically close but physically invalid spot. In this demo, several valid candidates remain so the later least-squares ranking is meaningful.


In [ ]:
result = engine.assign(vehicle)
feasible = [c for c in result.candidates if c.feasible]
infeasible = [c for c in result.candidates if not c.feasible]

print("Total spots:", len(result.candidates))
print("Feasible candidates:", len(feasible))
assert len(feasible) >= 3, "The main demo should keep multiple feasible candidates so optimization can be demonstrated."
print("Rejected candidates:", len(infeasible))
print("Feasible spot IDs:", [c.spot_id for c in feasible])


## 6. Stage 4 — Orthogonal projection

> **Important demo checkpoint:** this scenario keeps four feasible candidates, so the projection and ranking stages are not trivial.

For the feasible candidate matrix `X`, the project constructs:

\[ P = X^T(XX^T)^+X \]

Then the ideal vehicle demand is projected as:

\[ \hat v = Pv \]

The projection answers:

> **What is the closest representation of the desired vehicle requirements inside the feature space generated by the feasible parking spots?**


In [ ]:
print("Feasible matrix X shape:", result.spot_matrix.shape)
print("\nVehicle vector v:")
print(np.round(result.vehicle_vector, 3))
print("\nOrthogonal projector P:")
print(np.round(result.projector, 3))
print("\nProjected demand v_hat = P v:")
print(np.round(result.projected_vehicle_vector, 3))

projection_check = np.linalg.norm(result.projector @ result.projector - result.projector)
print("\nProjection check ||P^2 - P|| =", projection_check)
symmetry_check = np.linalg.norm(result.projector.T - result.projector)
print("Symmetry check ||P^T - P|| =", symmetry_check)
gap = result.vehicle_vector - result.projected_vehicle_vector
orthogonality_check = np.linalg.norm(result.spot_matrix @ gap)
print("Orthogonality check ||X(v - v_hat)|| =", orthogonality_check)


### What to say

> “Projection finds the closest point in the feasible feature subspace. We use the Moore–Penrose pseudoinverse because the candidate matrix can be rectangular or rank-deficient. The small value of `||P²-P||` confirms the projector property numerically.”


### Important interpretation for this main parking scenario

In this particular four-candidate parking configuration, the vehicle vector already lies in the row-space generated by the feasible spots. Therefore the orthogonal projection can legitimately return **v-hat = v** (up to floating-point rounding). That is not an error: an orthogonal projector leaves vectors that are already inside the subspace unchanged.

For a non-trivial projection where `v-hat` differs from `v`, use the **illustrative worked example in the PPT**; it uses a small rank-deficient matrix to demonstrate the same projection mathematics with a visible non-zero residual.

## Optional viva backup — a non-trivial projection

In the main parking scenario, the requested vehicle vector happens to already lie in the feasible row-space, so the projector returns `v-hat = v`. That is a valid projection result.

If Madam asks to see a case where projection actually changes the vector, run the following small illustrative example. It is the same mathematics used in the worked example in the PPT.

In [ ]:
X_demo = np.array([
    [0.43, 0.18, 1.00, 0.00, 0.67],
    [0.86, 0.18, 1.00, 0.00, 0.67],
    [0.29, 0.36, 1.00, 0.00, 1.00],
    [0.71, 0.36, 1.00, 0.00, 1.00],
], dtype=float)
v_demo = np.array([0.0, 0.0, 1.0, 0.0, 0.67])
P_demo = X_demo.T @ np.linalg.pinv(X_demo @ X_demo.T) @ X_demo
vhat_demo = P_demo @ v_demo
r_demo = v_demo - vhat_demo

print("Illustrative projected vector v_hat:", np.round(vhat_demo, 3))
print("Residual r:", np.round(r_demo, 3))
print("||P^2 - P|| =", np.linalg.norm(P_demo @ P_demo - P_demo))
print("||P^T - P|| =", np.linalg.norm(P_demo.T - P_demo))
print("||Xr|| =", np.linalg.norm(X_demo @ r_demo))

## 7. Stage 5 — Least-squares ranking

Because multiple feasible spots remain, we can compare their scores and show why the minimum wins.

For each feasible spot `x_i`, the project computes:

\[ LS_i = ||W(x_i-\hat v)||_2^2 \]

The lower the least-squares residual, the closer the parking spot is to the projected vehicle demand.

The final objective also includes normalized gate distance and a lateral geometric term:


In [ ]:
ranking = sorted(
    [c for c in result.candidates if c.feasible],
    key=lambda c: c.total_cost
)

ranking_df = pd.DataFrame([
    {
        "Rank": i + 1,
        "Spot": c.spot_id,
        "Least-squares": c.least_squares,
        "Gate distance": c.gate_distance,
        "Lateral offset": c.lateral_offset,
        "Total score": c.total_cost,
    }
    for i, c in enumerate(ranking)
])

display(ranking_df.round(4))
print("\nBest assignment:", result.assigned_spot.spot_id if result.assigned_spot else "NONE")


### What to say

> “Lower least-squares error means a better feature match. Because four feasible spaces remain in this demo, we can actually see the optimizer rank alternatives rather than ending with a single candidate.” The system then combines that with gate distance and lateral geometry. The minimum total score determines the final feasible assignment.”


## 8. Stage 6 — Visualize the result


In [ ]:
assigned_id = result.assigned_spot.spot_id if result.assigned_spot else None

plt.figure(figsize=(9, 6))
for s in engine.spots:
    if s.spot_id == assigned_id:
        marker, size = "*", 300
    elif s.ev and s.accessible:
        marker, size = "s", 150
    elif s.accessible:
        marker, size = "^", 140
    elif s.ev:
        marker, size = "s", 140
    else:
        marker, size = "o", 100
    plt.scatter(s.x, s.y, marker=marker, s=size)
    plt.text(s.x, s.y + 0.20, s.spot_id, ha="center", fontsize=8)

for gate in engine.gates.values():
    plt.scatter(gate.x, gate.y, marker="D", s=120)
    plt.text(gate.x, gate.y - 0.45, gate.gate_id, ha="center", fontsize=9)

plt.title(f"Smart Parking Assignment: {vehicle.vehicle_id} → {assigned_id}")
plt.xlabel("X position")
plt.ylabel("Y position")
plt.grid(True, alpha=0.25)
plt.axis("equal")
plt.show()


## 9. Stage 7 — Complete pipeline log

The same engine records the intermediate mathematical stages so the decision is explainable.


In [ ]:
print("\n".join(result.log_lines))


## 10. Stage 8 — Multi-vehicle simulation

Now we show that the system can process multiple vehicles sequentially and update occupancy after each assignment.


In [ ]:
engine.reset()
vehicles = [
    Vehicle("V01", "G1", False, False, "Compact"),
    Vehicle("V02", "G1", True, False, "Standard"),
    Vehicle("V03", "G2", False, True, "Standard"),
    Vehicle("V04", "G2", True, True, "Large"),
]

rows = []
for v in vehicles:
    r = engine.assign(v)
    rows.append({
        "Vehicle": v.vehicle_id,
        "Gate": v.entry_gate,
        "EV": v.ev_required,
        "Accessible": v.accessible_required,
        "Size": v.size,
        "Assigned spot": r.assigned_spot.spot_id if r.assigned_spot else "NONE",
    })

display(pd.DataFrame(rows))
print("\nOccupancy summary:", engine.occupancy_summary())


# Final viva cheat sheet

**What is X?** A matrix whose rows are feasible parking-space feature vectors.

**What is v?** A vehicle demand vector in the same five-dimensional feature space.

**Why projection?** To find the closest representable demand inside the feasible feature subspace.

**Why least squares?** The squared L2 residual provides a numerical measure of mismatch and lets us rank candidates.

**Why constraints first?** EV, accessibility, size and vacancy are non-negotiable.

**One-line explanation:**

> We represent parking spaces and vehicle requirements as vectors and matrices, project the vehicle demand into the feasible feature space, rank feasible spaces using weighted least squares plus gate-aware geometry, and assign the minimum-cost spot.
